# Experiments: Alternative Methods for Disaster Tweet Classification

This notebook collects alternative methods and experiments that were not included in the main project notebook. It is intended for further exploration, comparison, and ablation studies.

## 1. Baseline: Logistic Regression with TF-IDF
A simple baseline using TF-IDF features and logistic regression for classification.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

train = pd.read_csv('nlp-getting-started/train.csv')
X = train['text'].fillna("")
y = train['target']

vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = vectorizer.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_val)

print(classification_report(y_val, y_pred))
print(f"Validation Accuracy: {accuracy_score(y_val, y_pred):.4f}")

              precision    recall  f1-score   support

           0       0.80      0.88      0.84       874
           1       0.82      0.70      0.75       649

    accuracy                           0.81      1523
   macro avg       0.81      0.79      0.80      1523
weighted avg       0.81      0.81      0.80      1523

Validation Accuracy: 0.8056


## 2. Word2Vec Embeddings + Simple Neural Network
Train a simple neural network using averaged Word2Vec embeddings for each tweet.

In [4]:
from gensim.models import Word2Vec
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import numpy as np

# Tokenize tweets
tokens = [str(text).split() for text in train['text'].fillna("")]
word2vec = Word2Vec(sentences=tokens, vector_size=100, window=5, min_count=2, workers=4)

def get_w2v_vector(tokens, model, size=100):
    vec = np.zeros(size)
    count = 0
    for word in tokens:
        if word in model.wv:
            vec += model.wv[word]
            count += 1
    return vec / count if count > 0 else vec

X_w2v = np.array([get_w2v_vector(t, word2vec, 100) for t in tokens])

X_train, X_val, y_train, y_val = train_test_split(X_w2v, y, test_size=0.2, random_state=42)

model = Sequential([
    Dense(64, activation='relu', input_shape=(100,)),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))

Epoch 1/10


/Users/gabriele.gabrielli/Documents-personal/DisasterTweetsNLP_Kaggle_CUBoulder/.venv/lib/python3.10/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5346 - loss: 0.6969 - val_accuracy: 0.5739 - val_loss: 0.6789
Epoch 2/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5346 - loss: 0.6969 - val_accuracy: 0.5739 - val_loss: 0.6789
Epoch 2/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 736us/step - accuracy: 0.5728 - loss: 0.6803 - val_accuracy: 0.5739 - val_loss: 0.6752
Epoch 3/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 736us/step - accuracy: 0.5728 - loss: 0.6803 - val_accuracy: 0.5739 - val_loss: 0.6752
Epoch 3/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 620us/step - accuracy: 0.5708 - loss: 0.6798 - val_accuracy: 0.5739 - val_loss: 0.6734
Epoch 4/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 620us/step - accuracy: 0.5708 - loss: 0.6798 - val_accuracy: 0.5739 - val_loss: 0.6734
Epoch 4/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 618us/step - accuracy: 0.5605 - loss: 0.6777 - val_accuracy: 0.5739 - val_loss: 0.6719
Epoch 5/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 618us/step - accuracy: 0.5605 - loss: 0.6777 - val_accuracy: 0

## 3. LSTM Model with Embedding Layer
A simple LSTM-based model using Keras Embedding for sequence modeling.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Input
from tensorflow.keras.models import Model

max_words = 5000
max_len = 40

# Tokenization
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train['text'].fillna("").tolist())
sequences = tokenizer.texts_to_sequences(train['text'].fillna("").tolist())
X_seq = pad_sequences(sequences, maxlen=max_len)

X_train, X_val, y_train, y_val = train_test_split(X_seq, y, test_size=0.2, random_state=42)

input_layer = Input(shape=(max_len,))
embedding_layer = Embedding(input_dim=max_words, output_dim=64, input_length=max_len)(input_layer)
lstm_layer = LSTM(64)(embedding_layer)
output_layer = Dense(1, activation='sigmoid')(lstm_layer)
lstm_model = Model(inputs=input_layer, outputs=output_layer)
lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_val, y_val))

Epoch 1/5


/Users/gabriele.gabrielli/Documents-personal/DisasterTweetsNLP_Kaggle_CUBoulder/.venv/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.6506 - loss: 0.6168 - val_accuracy: 0.8037 - val_loss: 0.4384
Epoch 2/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.6506 - loss: 0.6168 - val_accuracy: 0.8037 - val_loss: 0.4384
Epoch 2/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8531 - loss: 0.3461 - val_accuracy: 0.7978 - val_loss: 0.4604
Epoch 3/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8531 - loss: 0.3461 - val_accuracy: 0.7978 - val_loss: 0.4604
Epoch 3/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9040 - loss: 0.2554 - val_accuracy: 0.8004 - val_loss: 0.4871
Epoch 4/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9040 - loss: 0.2554 - val_accuracy: 0.8004 - val_loss: 0.4871
Epoch 4/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9259 - loss: 0.2093 - val_accuracy: 0.7879 - val_loss: 0.5519
Epoch 5/5
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9259 - loss: 0.2093 - val_accuracy: 0.7879 - val

In [ ]:

# Predict on validation set
y_val_pred = lstm_model.predict(X_val)
y_val_pred_labels = (y_val_pred > 0.5).astype(int).flatten()

# Calculate metrics
accuracy = accuracy_score(y_val, y_val_pred_labels)
precision = precision_score(y_val, y_val_pred_labels)
recall = recall_score(y_val, y_val_pred_labels)
f1 = f1_score(y_val, y_val_pred_labels)
cm = confusion_matrix(y_val, y_val_pred_labels)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)

48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Accuracy:  0.7774
Precision: 0.7870
Recall:    0.6549
F1 Score:  0.7149
Confusion Matrix:
[[759 115]
 [224 425]]


## 4. Experiment Tracking and Notes
- Compare validation accuracy and F1-score across all methods.
- Note training time, overfitting, and any preprocessing differences.
- Use this notebook to try new architectures or preprocessing ideas before moving to the main notebook.